# Pakistani Law RAG — Data Collection, Chunking & Pinecone Upsert

**Run this notebook once.** It will:
1. Scrape Pakistani legal texts from public sources
2. Clean and chunk them two ways (Fixed + Recursive) for the ablation study
3. Embed all chunks with `all-MiniLM-L6-v2`
4. Upsert both chunk sets into Pinecone under separate namespaces
5. Save chunk JSONs locally (needed for BM25 in the app)

**Before running:** Make sure you have selected `Runtime > Change runtime type > T4 GPU`

---
## Cell 1 — Install Dependencies

In [1]:
!pip install -q pinecone sentence-transformers langchain langchain-community \
                 beautifulsoup4 requests pdfplumber rank-bm25 tqdm tiktoken \
                 lxml html5lib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source 

---
## Cell 2 — Configuration (EDIT THIS CELL)

In [8]:
# ============================================================
#  FILL IN YOUR OWN VALUES HERE
# ============================================================

PINECONE_API_KEY  = "YOUR_PINECONE_API_KEY"   # from pinecone.io dashboard
PINECONE_INDEX    = "pakistani-law"                 # the index name you created

# Namespaces — one per chunking strategy (for ablation study)
NS_FIXED     = "fixed"
NS_RECURSIVE = "recursive"

# Chunking parameters
FIXED_CHUNK_SIZE    = 256    # tokens
FIXED_CHUNK_OVERLAP = 30     # tokens
REC_CHUNK_SIZE      = 256    # characters (RecursiveCharacterTextSplitter uses chars)
REC_CHUNK_OVERLAP   = 50

# Embedding model — produces 384-dim vectors, matches your Pinecone index
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
EMBED_BATCH     = 64

print("Config loaded.")

Config loaded.


---
## Cell 3 — Import Libraries

In [9]:
import requests
import time
import json
import re
import os
from bs4 import BeautifulSoup
from tqdm import tqdm
import tiktoken
from sentence_transformers import SentenceTransformer
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone

print("All imports successful.")

All imports successful.


---
## Cell 4 — Define the Corpus (Pakistani Law Acts)

Each entry is a URL from `pakistancode.gov.pk` — the official public registry of Pakistani legislation.
We scrape the HTML directly. No login required, all public domain.

In [10]:
CORPUS = [
    # (display_name, year, url)
    # These URLs hit the Pakistan Code official site HTML viewer
    ("Constitution of Pakistan 1973",           1973, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJl="),
    ("Pakistan Penal Code 1860",                1860, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJm="),
    ("Code of Criminal Procedure 1898",         1898, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJn="),
    ("Civil Procedure Code 1908",               1908, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJo="),
    ("Contract Act 1872",                       1872, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJp="),
    ("Qanun-e-Shahadat Order 1984",             1984, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJq="),
    ("Anti-Terrorism Act 1997",                 1997, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJr="),
    ("National Accountability Ordinance 1999",  1999, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJs="),
    ("Muslim Family Laws Ordinance 1961",       1961, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJt="),
    ("Companies Act 2017",                      2017, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJu="),
    ("Income Tax Ordinance 2001",               2001, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJv="),
    ("Arbitration Act 1940",                    1940, "https://www.pakistancode.gov.pk/english/UY2FqaJw1-apaUY2Fqa-apaUY2FqaJJJJJJJJJJJw="),
]

# ----------------------------------------------------------------
# FALLBACK DIRECT TEXT SOURCES (more reliable than scraping)
# If the scraper fails for an act, we also define fallback plain-
# text URLs from commonlii.org which has clean HTML for Pakistani law
# ----------------------------------------------------------------
FALLBACK_CORPUS = [
    ("Pakistan Penal Code 1860",           1860, "https://www.commonlii.org/pk/legis/num_act/ppc1860132/"),
    ("Constitution of Pakistan 1973",      1973, "https://www.commonlii.org/pk/legis/const/2012/"),
    ("Contract Act 1872",                  1872, "https://www.commonlii.org/pk/legis/num_act/ca1872149/"),
    ("Code of Criminal Procedure 1898",    1898, "https://www.commonlii.org/pk/legis/num_act/cocp1898230/"),
    ("Qanun-e-Shahadat Order 1984",        1984, "https://www.commonlii.org/pk/legis/num_act/qeo1984268/"),
]

print(f"Corpus defined: {len(CORPUS)} primary + {len(FALLBACK_CORPUS)} fallback sources.")

Corpus defined: 12 primary + 5 fallback sources.


---
## Cell 5 — Scraper Functions

In [11]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

def clean_text(raw: str) -> str:
    """Remove excess whitespace, page markers, and non-printable chars."""
    # Collapse multiple newlines to double
    text = re.sub(r'\n{3,}', '\n\n', raw)
    # Remove page number patterns like 'Page 1 of 45'
    text = re.sub(r'Page\s+\d+\s+of\s+\d+', '', text, flags=re.IGNORECASE)
    # Remove lines that are just numbers (page numbers)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    # Remove excessive spaces
    text = re.sub(r' {3,}', ' ', text)
    # Strip leading/trailing whitespace
    text = text.strip()
    return text


def scrape_commonlii(url: str, name: str, year: int) -> dict | None:
    """
    Scrape a legal document from commonlii.org.
    Returns a document dict or None on failure.
    """
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'lxml')

        # commonlii puts the main content inside <div class="judgment"> or <body>
        # Try judgment div first, fall back to body
        content = soup.find('div', class_='judgment')
        if not content:
            content = soup.find('body')
        if not content:
            return None

        # Remove navigation, scripts, styles
        for tag in content.find_all(['script', 'style', 'nav', 'header', 'footer', 'noscript']):
            tag.decompose()

        raw_text = content.get_text(separator='\n')
        text = clean_text(raw_text)

        if len(text) < 1000:   # Too short — probably a failed scrape
            return None

        return {
            "name":   name,
            "year":   year,
            "url":    url,
            "text":   text,
            "length": len(text)
        }

    except Exception as e:
        print(f"  FAILED ({name}): {e}")
        return None


def scrape_pakistancode(url: str, name: str, year: int) -> dict | None:
    """
    Scrape from pakistancode.gov.pk.
    The site renders content inside <div id='content'> or similar.
    """
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'lxml')

        # Try common content containers
        content = (soup.find('div', id='content') or
                   soup.find('div', class_='content') or
                   soup.find('div', class_='act-content') or
                   soup.find('main') or
                   soup.find('article'))

        if not content:
            content = soup.find('body')

        for tag in content.find_all(['script', 'style', 'nav', 'header', 'footer']):
            tag.decompose()

        raw_text = content.get_text(separator='\n')
        text = clean_text(raw_text)

        if len(text) < 1000:
            return None

        return {
            "name":   name,
            "year":   year,
            "url":    url,
            "text":   text,
            "length": len(text)
        }

    except Exception as e:
        print(f"  FAILED ({name}): {e}")
        return None


print("Scraper functions defined.")

Scraper functions defined.


---
## Cell 6 — Run the Scraper

This first tries `commonlii.org` (cleaner HTML), then falls back to `pakistancode.gov.pk`.
Be patient — it sleeps 1-2s between requests to avoid being blocked.

In [12]:
documents = []
failed    = []

print("Scraping fallback corpus (commonlii.org — cleaner HTML)...")
for name, year, url in tqdm(FALLBACK_CORPUS, desc="Scraping"):
    doc = scrape_commonlii(url, name, year)
    if doc:
        documents.append(doc)
        print(f"  OK  {name} — {doc['length']:,} chars")
    else:
        failed.append((name, year, url))
        print(f"  FAIL {name}")
    time.sleep(1.5)

# For documents that failed, try pakistancode.gov.pk
if failed:
    print(f"\nRetrying {len(failed)} failed docs on pakistancode.gov.pk...")
    for name, year, url in failed:
        # Find matching URL in primary corpus
        match = next((u for n, y, u in CORPUS if n == name), None)
        if match:
            doc = scrape_pakistancode(match, name, year)
            if doc:
                documents.append(doc)
                print(f"  OK  {name} — {doc['length']:,} chars")
        time.sleep(2)

print(f"\n=== Scraped {len(documents)} documents ===")
total_chars = sum(d['length'] for d in documents)
print(f"Total text: {total_chars:,} characters (~{total_chars//5:,} words)")

# Save raw docs as backup
with open("raw_documents.json", "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)
print("Raw documents saved to raw_documents.json")

Scraping fallback corpus (commonlii.org — cleaner HTML)...


Scraping:   0%|          | 0/5 [00:00<?, ?it/s]

  FAILED (Pakistan Penal Code 1860): 410 Client Error: Gone for url: https://www.commonlii.org/pk/legis/num_act/ppc1860132/
  FAIL Pakistan Penal Code 1860


Scraping:  20%|██        | 1/5 [00:02<00:11,  2.94s/it]

  FAILED (Constitution of Pakistan 1973): 410 Client Error: Gone for url: https://www.commonlii.org/pk/legis/const/2012/
  FAIL Constitution of Pakistan 1973


...

Scraping:  80%|████████  | 4/5 [00:19<00:05,  5.78s/it]

  FAILED (Qanun-e-Shahadat Order 1984): 410 Client Error: Gone for url: https://www.commonlii.org/pk/legis/num_act/qeo1984268/
  FAIL Qanun-e-Shahadat Order 1984


Scraping: 100%|██████████| 5/5 [00:22<00:00,  4.46s/it]



Retrying 5 failed docs on pakistancode.gov.pk...


KeyboardInterrupt: 

---
## Cell 7 — Manual Text Entry (Run if scraping failed badly)

If you got fewer than 3 documents from the scraper, run this cell.
It loads from text files you upload manually.

**Instructions:**
1. Go to `pakistancode.gov.pk`, open any act, Ctrl+A, Ctrl+C, paste into a .txt file
2. Upload the .txt files to this Colab session using the Files panel on the left
3. Run this cell — it will load whatever .txt files are present

In [35]:
import glob

txt_files = glob.glob("*.txt")
if txt_files:
    for path in txt_files:
        name = os.path.splitext(os.path.basename(path))[0].replace('_', ' ')
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            text = clean_text(f.read())
        if len(text) > 500:
            doc = {"name": name, "year": 0, "url": path, "text": text, "length": len(text)}
            # Avoid duplicates
            existing_names = [d['name'] for d in documents]
            if name not in existing_names:
                documents.append(doc)
                print(f"Loaded from file: {name} — {len(text):,} chars")
else:
    print("No .txt files found. Skipping manual entry.")

print(f"\nTotal documents available: {len(documents)}")


Total documents available: 9


In [38]:
## BW 7 and 8, FIX LINE BREAKS

import re

def fix_pdf_linebreaks(text):
    """Fix haphazard PDF line breaks from copy-paste."""
    # Protect real paragraph breaks first
    text = re.sub(r'\n\n+', '<<PARA>>', text)

    lines = text.split('\n')
    joined = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if joined:
            prev = joined[-1]
            ends_sentence  = prev.endswith(('.', ':', ';', ')', ']', '—'))
            starts_section = bool(re.match(r'^\d+[\.\)]\s', line))
            is_heading     = line.isupper() and len(line) < 80
            starts_with_cap_after_short = (
                len(prev) < 40 and line[0].isupper()
            )
            if ends_sentence or starts_section or is_heading or starts_with_cap_after_short:
                joined.append(line)
            else:
                joined[-1] = prev + ' ' + line
        else:
            joined.append(line)

    text = '\n'.join(joined)
    text = text.replace('<<PARA>>', '\n\n')
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

# Apply fix to all loaded documents
for doc in documents:
    original_len = len(doc['text'])
    doc['text']  = fix_pdf_linebreaks(doc['text'])
    new_len      = len(doc['text'])
    print(f"{doc['name'][:45]:<45} {original_len:>8,} → {new_len:>8,} chars")

# Save fixed documents as backup
with open("raw_documents_fixed.json", "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print(f"\nFixed {len(documents)} documents.")
print("Now run Cell 8 onwards.")

muslim family laws ordinance 1961               15,725 →   15,725 chars
code of criminal procedure 1898                763,347 →  763,347 chars
national accountability ordinance 1999          99,941 →   99,941 chars
qanun e shahadat 1984                          777,776 →  777,776 chars
contract act 1872                              160,694 →  160,694 chars
code of civil procedure 1908                   777,776 →  777,776 chars
pakistan penal code 1860                       495,771 →  495,771 chars
companies act 2017                             920,271 →  920,271 chars
constitution of pakistan 1973                  439,767 →  439,767 chars

Fixed 9 documents.
Now run Cell 8 onwards.


---
## Cell 8 — Chunking Strategy A: Fixed Size (Token-Based)

Uses `tiktoken` to count tokens accurately. Slides a window of 256 tokens with 30-token overlap.
This is the baseline — simple, predictable, ignores sentence boundaries.

In [39]:
enc = tiktoken.get_encoding("cl100k_base")  # Same tokenizer used by many models

def fixed_chunk(text: str, chunk_size: int = 256, overlap: int = 30) -> list[str]:
    """
    Split text into chunks of exactly `chunk_size` tokens with `overlap` token overlap.
    Returns list of text strings.
    """
    tokens = enc.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = enc.decode(chunk_tokens)
        if len(chunk_text.strip()) > 50:  # skip tiny fragments
            chunks.append(chunk_text.strip())
        start += chunk_size - overlap
    return chunks


fixed_chunks = []
for doc in documents:
    doc_chunks = fixed_chunk(doc['text'], FIXED_CHUNK_SIZE, FIXED_CHUNK_OVERLAP)
    for i, chunk_text in enumerate(doc_chunks):
        fixed_chunks.append({
            "id":       f"fixed-{doc['name'][:30].replace(' ', '-')}-{i:04d}",
            "text":     chunk_text,
            "source":   doc['name'],
            "year":     doc['year'],
            "url":      doc['url'],
            "strategy": "fixed",
            "chunk_idx": i
        })

# Clean IDs — Pinecone IDs cannot have special chars except - and _
for c in fixed_chunks:
    c['id'] = re.sub(r'[^a-zA-Z0-9\-_]', '', c['id'])

print(f"Fixed chunking produced: {len(fixed_chunks)} chunks")
print(f"Sample chunk:\n---\n{fixed_chunks[0]['text'][:300]}\n---")

with open("chunks_fixed.json", "w", encoding="utf-8") as f:
    json.dump(fixed_chunks, f, ensure_ascii=False, indent=2)
print("Saved to chunks_fixed.json")

Fixed chunking produced: 4527 chunks
Sample chunk:
---
THE MUSLIM FAMILY LAWS ORDINANCE, 1961
CONTENTS __________
SECTIONS:
1. Short title, extent, application and commencement.
2. Definitions.
3. Ordinance to override other laws etc.
4. Succession.
5. Registration of marriages.
6. Polygamy.
7. Talaq.
8. Dissolution of marriage otherwise than by talaq.

---
Saved to chunks_fixed.json


---
## Cell 9 — Chunking Strategy B: Recursive (Structure-Aware)

Splits on natural boundaries in order: paragraph breaks → newlines → sentences → words.
Only falls to the next level if the current chunk is still too large.
Produces more semantically coherent chunks that respect section boundaries.

In [40]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""],
    chunk_size=1200,       # ~256 tokens in characters
    chunk_overlap=150,
    length_function=len,
)

recursive_chunks = []
for doc in documents:
    doc_chunks = splitter.split_text(doc['text'])
    for i, chunk_text in enumerate(doc_chunks):
        if len(chunk_text.strip()) < 50:
            continue
        recursive_chunks.append({
            "id":       f"rec-{doc['name'][:30].replace(' ', '-')}-{i:04d}",
            "text":     chunk_text.strip(),
            "source":   doc['name'],
            "year":     doc['year'],
            "url":      doc['url'],
            "strategy": "recursive",
            "chunk_idx": i
        })

for c in recursive_chunks:
    c['id'] = re.sub(r'[^a-zA-Z0-9\-_]', '', c['id'])

print(f"Recursive chunking produced: {len(recursive_chunks)} chunks")
print(f"Sample chunk:\n---\n{recursive_chunks[0]['text'][:300]}\n---")

with open("chunks_recursive.json", "w", encoding="utf-8") as f:
    json.dump(recursive_chunks, f, ensure_ascii=False, indent=2)
print("Saved to chunks_recursive.json")

print(f"\n=== Chunk Summary ===")
print(f"Fixed chunks:     {len(fixed_chunks)}")
print(f"Recursive chunks: {len(recursive_chunks)}")
print(f"Total vectors to upsert: {len(fixed_chunks) + len(recursive_chunks)}")

Recursive chunking produced: 5340 chunks
Sample chunk:
---
THE MUSLIM FAMILY LAWS ORDINANCE, 1961
CONTENTS __________
SECTIONS:
1. Short title, extent, application and commencement.
2. Definitions.
3. Ordinance to override other laws etc.
4. Succession.
5. Registration of marriages.
6. Polygamy.
7. Talaq.
8. Dissolution of marriage otherwise than by talaq.

---
Saved to chunks_recursive.json

=== Chunk Summary ===
Fixed chunks:     4527
Recursive chunks: 5340
Total vectors to upsert: 9867


---
## Cell 10 — Load Embedding Model

Downloads `all-MiniLM-L6-v2` (~80MB). Uses GPU if available.

In [41]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

embed_model = SentenceTransformer(EMBEDDING_MODEL, device=device)

# Quick sanity check
test_vec = embed_model.encode("Test sentence for Pakistani law system.")
print(f"Embedding model loaded. Output dimension: {len(test_vec)} (expected 384)")

Using device: cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded. Output dimension: 384 (expected 384)


---
## Cell 11 — Generate Embeddings for Both Chunk Sets

In [42]:
def embed_chunks(chunks: list[dict], batch_size: int = 64) -> list[list[float]]:
    """Embed all chunks and return as list of float lists."""
    texts = [c['text'] for c in chunks]
    embeddings = embed_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    return embeddings.tolist()


print("Embedding fixed chunks...")
fixed_embeddings = embed_chunks(fixed_chunks, EMBED_BATCH)
print(f"Done. Shape: {len(fixed_embeddings)} x {len(fixed_embeddings[0])}")

print("\nEmbedding recursive chunks...")
recursive_embeddings = embed_chunks(recursive_chunks, EMBED_BATCH)
print(f"Done. Shape: {len(recursive_embeddings)} x {len(recursive_embeddings[0])}")

Embedding fixed chunks...


Batches:   0%|          | 0/71 [00:00<?, ?it/s]

Done. Shape: 4527 x 384

Embedding recursive chunks...


Batches:   0%|          | 0/84 [00:00<?, ?it/s]

Done. Shape: 5340 x 384


---
## Cell 12 — Connect to Pinecone

In [21]:
pc    = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX)

stats = index.describe_index_stats()
print(f"Connected to Pinecone index: '{PINECONE_INDEX}'")
print(f"Current vector count: {stats.total_vector_count}")
print(f"Namespaces present:   {list(stats.namespaces.keys())}")

Connected to Pinecone index: 'pakistani-law'
Current vector count: 10196
Namespaces present:   ['fixed', 'recursive']


In [34]:
## Reset EMBEDDINGS, DELETE PINECOME DATA

# Clear existing data — paste this as a cell just before the upsert cell
print("Clearing existing Pinecone namespaces...")
index.delete(delete_all=True, namespace="fixed")
index.delete(delete_all=True, namespace="recursive")
import time
time.sleep(5)  # wait for deletion to propagate
stats = index.describe_index_stats()
print(f"Vectors remaining: {stats.total_vector_count}")
print("Ready to upsert fresh data.")

Clearing existing Pinecone namespaces...
Vectors remaining: 0
Ready to upsert fresh data.


In [43]:
## TOC filter cell

def is_toc_chunk(text):
    """
    Detect table of contents chunks.
    TOC chunks have many lines that are just 'number. title' patterns
    with very little actual prose.
    """
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if len(lines) < 3:
        return False

    # Count lines that look like TOC entries: "302. Punishment of..."
    toc_pattern = re.compile(r'^\d+[\.\)]\s+[A-Za-z]')
    toc_lines = sum(1 for l in lines if toc_pattern.match(l))

    # If more than 50% of lines are TOC entries — it's a TOC chunk
    return (toc_lines / len(lines)) > 0.5


def is_useful_chunk(text):
    """Filter out chunks that won't help answer legal questions."""
    # Too short
    if len(text.strip()) < 100:
        return False

    # Pure amendment/substitution notes
    amendment_indicators = [
        'subs. by', 'ins. by', 'rep. by', 'omitted by',
        'substituted by', 'inserted by', 'repealed by',
        'a.o., 19', 'ord.', 's.2 and sch'
    ]
    text_lower = text.lower()
    amendment_count = sum(1 for ind in amendment_indicators
                         if ind in text_lower)
    # If chunk is mostly amendment notes
    if amendment_count > 3 and len(text) < 400:
        return False

    # TOC chunk
    if is_toc_chunk(text):
        return False

    return True


# Apply filter to both chunk sets
print("Before filtering:")
print(f"  Fixed chunks:     {len(fixed_chunks)}")
print(f"  Recursive chunks: {len(recursive_chunks)}")

fixed_chunks_clean     = [c for c in fixed_chunks     if is_useful_chunk(c['text'])]
recursive_chunks_clean = [c for c in recursive_chunks if is_useful_chunk(c['text'])]

print("\nAfter filtering:")
print(f"  Fixed chunks:     {len(fixed_chunks_clean)} (removed {len(fixed_chunks)-len(fixed_chunks_clean)})")
print(f"  Recursive chunks: {len(recursive_chunks_clean)} (removed {len(recursive_chunks)-len(recursive_chunks_clean)})")

# Save filtered versions — these replace the old chunk files everywhere
with open("chunks_fixed.json", "w", encoding="utf-8") as f:
    json.dump(fixed_chunks_clean, f, ensure_ascii=False, indent=2)
with open("chunks_recursive.json", "w", encoding="utf-8") as f:
    json.dump(recursive_chunks_clean, f, ensure_ascii=False, indent=2)

print("\nSaved filtered chunk files.")

Before filtering:
  Fixed chunks:     4527
  Recursive chunks: 5340

After filtering:
  Fixed chunks:     4138 (removed 389)
  Recursive chunks: 4768 (removed 572)

Saved filtered chunk files.


---
## Cell 13 — Upsert to Pinecone

Upserts in batches of 100 (Pinecone's recommended batch size).
Metadata stored alongside each vector includes the raw chunk text —
this is what gets retrieved and shown to the LLM.

In [49]:
def upsert_to_pinecone(chunks: list[dict],
                       embeddings: list[list[float]],
                       namespace: str,
                       batch_size: int = 100):
    """
    Upsert chunks + embeddings into a Pinecone namespace.
    Metadata includes text, source, year, url, strategy.
    """
    total   = len(chunks)
    upserted = 0

    for start in tqdm(range(0, total, batch_size), desc=f"Upserting [{namespace}]"):
        end   = min(start + batch_size, total)
        batch = []

        for i in range(start, end):
            chunk = chunks[i]
            # Pinecone metadata values must be str, int, float, or bool
            # Truncate text to 40960 chars (Pinecone metadata limit)
            metadata = {
                "text":      chunk['text'][:40000],
                "source":    chunk['source'],
                "year":      int(chunk['year']),
                "url":       chunk['url'],
                "strategy":  chunk['strategy'],
                "chunk_idx": int(chunk['chunk_idx'])
            }
            batch.append({
                "id":       chunk['id'],
                "values":   embeddings[i],
                "metadata": metadata
            })

        index.upsert(vectors=batch, namespace=namespace)
        upserted += len(batch)

    print(f"Upserted {upserted} vectors into namespace '{namespace}'")


# --- Upsert fixed chunks ---
print("=" * 50)
upsert_to_pinecone(fixed_chunks, fixed_embeddings, NS_FIXED)

# --- Upsert recursive chunks ---
print("=" * 50)
upsert_to_pinecone(recursive_chunks, recursive_embeddings, NS_RECURSIVE)

Upserting [fixed]: 100%|██████████| 42/42 [00:18<00:00,  2.22it/s]


Upserted 4138 vectors into namespace 'fixed'


Upserting [recursive]: 100%|██████████| 48/48 [00:20<00:00,  2.30it/s]

Upserted 4768 vectors into namespace 'recursive'


---
## Cell 14 — Verify the Upsert

In [45]:
time.sleep(5)  # Wait for Pinecone to index

stats = index.describe_index_stats()
print("=== Pinecone Index Stats ===")
print(f"Total vectors: {stats.total_vector_count}")
for ns, ns_stats in stats.namespaces.items():
    print(f"  Namespace '{ns}': {ns_stats.vector_count} vectors")

=== Pinecone Index Stats ===
Total vectors: 9867
  Namespace 'recursive': 5340 vectors
  Namespace 'fixed': 4527 vectors


In [46]:
print("Deleting old namespaces...")
index.delete(delete_all=True, namespace="fixed")
index.delete(delete_all=True, namespace="recursive")
import time
time.sleep(5)
stats = index.describe_index_stats()
print(f"Vectors remaining: {stats.total_vector_count}")

Deleting old namespaces...
Vectors remaining: 0


In [47]:
import json

with open("chunks_fixed.json", "r", encoding="utf-8") as f:
    fixed_chunks = json.load(f)
with open("chunks_recursive.json", "r", encoding="utf-8") as f:
    recursive_chunks = json.load(f)

print(f"Fixed chunks loaded:     {len(fixed_chunks)}")
print(f"Recursive chunks loaded: {len(recursive_chunks)}")
# Should show 4138 and 4768

Fixed chunks loaded:     4138
Recursive chunks loaded: 4768


In [48]:
print("Embedding fixed chunks...")
fixed_embeddings = embed_chunks(fixed_chunks, EMBED_BATCH)
print(f"Done: {len(fixed_embeddings)} embeddings")

print("Embedding recursive chunks...")
recursive_embeddings = embed_chunks(recursive_chunks, EMBED_BATCH)
print(f"Done: {len(recursive_embeddings)} embeddings")

Embedding fixed chunks...


Batches:   0%|          | 0/65 [00:00<?, ?it/s]

Done: 4138 embeddings
Embedding recursive chunks...


Batches:   0%|          | 0/75 [00:00<?, ?it/s]

Done: 4768 embeddings


In [50]:
time.sleep(5)
stats = index.describe_index_stats()
print(f"Total vectors: {stats.total_vector_count}")
for ns, ns_stats in stats.namespaces.items():
    print(f"  '{ns}': {ns_stats.vector_count} vectors")

# Should show: fixed=4138, recursive=4768

Total vectors: 8906
  'recursive': 4768 vectors
  'fixed': 4138 vectors


---
## Cell 15 — Quick Retrieval Test

Run a sample query to verify the pipeline is working end to end before moving on.

In [51]:
test_query = "What is the punishment for murder under Pakistani law?"

query_vec = embed_model.encode(test_query).tolist()

results = index.query(
    vector=query_vec,
    top_k=3,
    namespace=NS_FIXED,
    include_metadata=True
)

print(f"Query: {test_query}\n")
print("Top 3 results from Pinecone (fixed namespace):")
print("=" * 60)
for i, match in enumerate(results.matches):
    print(f"\n[{i+1}] Score: {match.score:.4f}")
    print(f"     Source: {match.metadata['source']}")
    print(f"     Text preview: {match.metadata['text'][:250]}...")

Query: What is the punishment for murder under Pakistani law?

Top 3 results from Pinecone (fixed namespace):

[1] Score: 0.6899
     Source: code of criminal procedure 1898
     Text preview: under section 241 of the Pakistan Penal Code, and within the cognizance of the Court of Session (or High Court].
(c) And I hereby direct that you be tried by the said Court on the said charge.
[Signature and seal of the Magistrate]
[To be substituted...

[2] Score: 0.6686
     Source: pakistan penal code 1860
     Text preview: any place without and beyond Pakistan ;]; (2) 2 [* * * * * * *]
(3) 3 [* * * * * * *]

[(4) any person on any ship or aircraft registered in 5 [Pakistan] wherever it may be.].
Explanation. In this section the word "offence" includes every act committ...

[3] Score: 0.6627
     Source: pakistan penal code 1860
     Text preview: description for a term not exceeding six months, or with fine or with both.]
CHAPTER VI
OF OFFENCES AGAINST THE STATE
121. Waging or attempting to 

In [53]:
from sentence_transformers import SentenceTransformer
bi_encoder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
print("Bi-encoder loaded.")

test_queries = [
    "What is the punishment for murder under Pakistani law?",
    "qatl-i-amd punishment death imprisonment",
    "punishment for qatl Section 302",
    "whoever commits qatl-i-amd shall be punished",
]

for q in test_queries:
    results = index.query(
        vector=bi_encoder.encode(q).tolist(),
        top_k=3,
        namespace="fixed",
        include_metadata=True
    )
    top = results.matches[0] if results.matches else None
    if top:
        print(f"Q: {q}")
        print(f"   Score: {top.score:.4f} | {top.metadata['source']}")
        print(f"   Text:  {top.metadata['text'][:150]}")
        print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Bi-encoder loaded.
Q: What is the punishment for murder under Pakistani law?
   Score: 0.6899 | code of criminal procedure 1898
   Text:  under section 241 of the Pakistan Penal Code, and within the cognizance of the Court of Session (or High Court].
(c) And I hereby direct that you be t

Q: qatl-i-amd punishment death imprisonment
   Score: 0.7624 | pakistan penal code 1860
   Text:  ordinary course of nature is likely to cause death, or with the knowledge that his act is so imminently dangerous that it must in all probability caus

Q: punishment for qatl Section 302
   Score: 0.6225 | pakistan penal code 1860
   Text:  less than the value of thirty thousand six hundred and thirty grams of silver.
(2) For the purpose of sub-section (1), the Federal Government shall, b

Q: whoever commits qatl-i-amd shall be punished
   Score: 0.7062 | pakistan penal code 1860
   Text:  .
Illustration
A in order to cause hurt strikes Z with a stick or stone which in the ordinary course of nature is not

In [26]:
# Search directly in your chunk text for Section 302
hits = []
for chunk in fixed_chunks:
    if '302' in chunk['text'] and chunk['source'] == 'pakistan penal code 1860':
        hits.append(chunk)

print(f"Chunks containing '302' in PPC: {len(hits)}")
for h in hits[:3]:
    print(f"\n--- chunk_idx={h['chunk_idx']} ---")
    print(h['text'][:400])

Chunks containing '302' in PPC: 6

--- chunk_idx=22 ---
Muslim or preaching or propagating his faith
CHAPTER XVI
OF OFFENCES AFFECTING THE HUMAN BODY
Of Offences affecting Life
299. Definitions
300. Qatl-i-amd
301. Causing death of person other than the person whose death was intended
302. Punishment of qatl-i-amd
303. Qatl committed under ikrah-i-tam or ikrah-i-naqis
304. Proof of qatl-i-amd liable to qisas, etc
305. Wali
306. Qatl-i-amd not liable to

--- chunk_idx=194 ---
mdt. Act, 1939 (22 of 1939), s. 2.
3Rep. by Act 17 of 1862.

[Explanation.__ In section 176 and in this section the word “offence” includes any act committed at any place out of 2 [Pakistan], which, if committed in 2 [Pakistan], would be punishable under any of the following sections, namely, 302, 304, 382, 392, 393, 394, 395, 396, 397, 398, 399, 402, 435, 436, 449, 450, 457, 458, 459 and 460 ; an

--- chunk_idx=215 ---
has murdered Z, assists B to hide the body with the intention of screening B from punishment. A 

In [28]:
qatl_hits = [c for c in fixed_chunks
             if 'qatl' in c['text'].lower()
             and c['source'] == 'pakistan penal code 1860']
print(f"Chunks containing 'qatl' in PPC: {len(qatl_hits)}")
for h in qatl_hits[:2]:
    print(f"\n--- chunk_idx={h['chunk_idx']} ---")
    print(h['text'][:400])

Chunks containing 'qatl' in PPC: 26

--- chunk_idx=22 ---
Muslim or preaching or propagating his faith
CHAPTER XVI
OF OFFENCES AFFECTING THE HUMAN BODY
Of Offences affecting Life
299. Definitions
300. Qatl-i-amd
301. Causing death of person other than the person whose death was intended
302. Punishment of qatl-i-amd
303. Qatl committed under ikrah-i-tam or ikrah-i-naqis
304. Proof of qatl-i-amd liable to qisas, etc
305. Wali
306. Qatl-i-amd not liable to

--- chunk_idx=23 ---
anni or swara

311. Ta'zir after waiver or compounding of right of qisas in qatl-i-amd
312. Qatl-i-'amd after waiver or compounding of qisas
313. Right of qisas in qatl-i-amd
314. Execution of qisas in qatl-i-amd
315. Qatl shibh-i-'amd
316. Punishment for qatl shibh-i-amd
317. Person committing qatl debarred from succession
318. Qatl-i-khata
319. Punishment for qatl-i-khata
320. Punishment for qat


---
## Cell 16 — Download Your Chunk Files

Download both JSON files to your local machine. You will need them when building the Gradio app
(BM25 runs from these files — it is not stored in Pinecone).

In [63]:
from google.colab import files

print("Downloading chunk files...")
files.download("chunks_fixed.json")
files.download("chunks_recursive.json")
files.download("raw_documents.json")
print("Done. Save these files — you will upload them to HF Spaces later.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

FileNotFoundError: Cannot find file: raw_documents.json

---
## Cell 17 — Summary

Run this at the end to confirm everything is in order.

In [ ]:
print("╔══════════════════════════════════════════════════╗")
print("║         DATA & EMBEDDINGS — COMPLETE             ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  Documents scraped:       {len(documents):<24}║")
print(f"║  Fixed chunks:            {len(fixed_chunks):<24}║")
print(f"║  Recursive chunks:        {len(recursive_chunks):<24}║")
print(f"║  Pinecone index:          {PINECONE_INDEX:<24}║")
print(f"║  Namespace [fixed]:       {NS_FIXED:<24}║")
print(f"║  Namespace [recursive]:   {NS_RECURSIVE:<24}║")
print("╠══════════════════════════════════════════════════╣")
print("║  Next step: build retrieval pipeline             ║")
print("║  (BM25 + Semantic + RRF + CrossEncoder)          ║")
print("╚══════════════════════════════════════════════════╝")

╔══════════════════════════════════════════════════╗
║         DATA & EMBEDDINGS — COMPLETE             ║
╠══════════════════════════════════════════════════╣
║  Documents scraped:       10                      ║
║  Fixed chunks:            4875                    ║
║  Recursive chunks:        5321                    ║
║  Pinecone index:          pakistani-law           ║
║  Namespace [fixed]:       fixed                   ║
║  Namespace [recursive]:   recursive               ║
╠══════════════════════════════════════════════════╣
║  Next step: build retrieval pipeline             ║
║  (BM25 + Semantic + RRF + CrossEncoder)          ║
╚══════════════════════════════════════════════════╝


In [54]:
# Find the exact chunk with the punishment text
for chunk in fixed_chunks:
    if ('qatl-i-amd' in chunk['text'].lower() and
        'death' in chunk['text'].lower() and
        'pakistan penal code' in chunk['source'].lower()):
        print(f"chunk_idx={chunk['chunk_idx']}")
        print(chunk['text'][:600])
        print("---")

chunk_idx=293
pretext of honour” means an offence committed in the name or on the pretext of karo kari, siyah kari or similar other customs or practices;]
(j) “qatl” (قتل (means causing death of a person; (k) “qisas” (قصاص (means punishment by causing similar hurt at the same part of the body of the convict as he has caused to the victim or by causing his death if he has committed qatl-i-amd, in exercise of the right of the victim or a wali ; (l) “ta’zir” (تعزیر (means punishment other than qisas (قصاص(, diyat (دیت(, arsh (ارش (or daman (ضمان ; (and (m) “wali” (ولی (means a person entitled to claim qisas.
3
---
chunk_idx=294
ordinary course of nature is likely to cause death, or with the knowledge that his act is so imminently dangerous that it must in all probability cause death, causes the death of such person, is said to commit qatl-e-amd.
301. Causing death of person other than the person whose death was intended. Where a person, by doing anything which he intends or knows to be li

In [56]:
import numpy as np

# Manually embed the chunk text and check similarity with query
target_chunks = [c for c in fixed_chunks
                 if 'qatl-i-amd' in c['text'].lower()
                 and 'death' in c['text'].lower()
                 and 'pakistan penal code' in c['source'].lower()]

if target_chunks:
    query_vec = bi_encoder.encode("punishment for murder under Pakistani law")

    for tc in target_chunks[:3]:
        chunk_vec = bi_encoder.encode(tc['text'])
        # cosine similarity
        sim = float(np.dot(query_vec, chunk_vec) /
                   (np.linalg.norm(query_vec) * np.linalg.norm(chunk_vec)))
        print(f"chunk_idx={tc['chunk_idx']} | similarity={sim:.4f}")
        print(tc['text'][:300])
        print("---")
else:
    print("NO chunks found with both 'qatl-i-amd' and 'death' in PPC")
    print("The actual punishment text is missing from your corpus.")

    # Check what qatl chunks DO exist
    qatl_chunks = [c for c in fixed_chunks
                   if 'qatl' in c['text'].lower()
                   and 'pakistan penal code' in c['source'].lower()]
    print(f"\nTotal qatl chunks in PPC: {len(qatl_chunks)}")
    for qc in qatl_chunks[:5]:
        print(f"  idx={qc['chunk_idx']}: {qc['text'][:150]}")

chunk_idx=293 | similarity=0.4352
pretext of honour” means an offence committed in the name or on the pretext of karo kari, siyah kari or similar other customs or practices;]
(j) “qatl” (قتل (means causing death of a person; (k) “qisas” (قصاص (means punishment by causing similar hurt at the same part of the body of the convict as he
---
chunk_idx=294 | similarity=0.3906
ordinary course of nature is likely to cause death, or with the knowledge that his act is so imminently dangerous that it must in all probability cause death, causes the death of such person, is said to commit qatl-e-amd.
301. Causing death of person other than the person whose death was intended. W
---
chunk_idx=297 | similarity=0.2803
-i-amd not liable to qisas.__ Qatl-i-amd shall not be liable to qisas in the following cases, namely:__ (a) when an offender is a minor or insane:
Provided that, where a person liable to qisas associates himself in the commission of the offence with a person not liable to qisas with the 

In [61]:
# !pip install groq


def rewrite_query(query):
    """Rewrite query using Pakistani legal terminology via Groq."""
    from groq import Groq
    groq = Groq(api_key="YOUR_GROQ_API_KEY")

    prompt = f"""You are a Pakistani legal expert.
Rewrite this question using precise Pakistani legal terminology.
Include both English terms AND specific legal terms used in Pakistani law.
Return ONLY the rewritten query, nothing else. Maximum 20 words.

Examples:
"murder punishment" → "qatl-i-amd punishment death imprisonment qisas diyat Section 302"
"theft penalty" → "theft dishonestly takes property Section 378 379 PPC"
"terrorism definition" → "terrorist act scheduled offence Anti-Terrorism Act"
"contract valid" → "agreement offer acceptance consideration lawful object contract"
"fundamental rights" → "fundamental rights Article 8 9 10 Constitution Pakistan"

Question: {query}
Rewritten:"""

    resp = groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=40,
        temperature=0.0
    )
    return resp.choices[0].message.content.strip()


# Test it
original = "What is the punishment for murder under Pakistani law?"
rewritten = rewrite_query(original)
print(f"Original:  {original}")
print(f"Rewritten: {rewritten}")

# Now retrieve with rewritten query
query_vec = bi_encoder.encode(rewritten).tolist()
results = index.query(
    vector=query_vec,
    top_k=3,
    namespace="fixed",
    include_metadata=True
)

print(f"\nResults with rewritten query:")
for i, m in enumerate(results.matches):
    print(f"[{i+1}] Score={m.score:.4f} | {m.metadata['source']}")
    print(f"     {m.metadata['text'][:200]}")
    print()

Original:  What is the punishment for murder under Pakistani law?
Rewritten: Qatl-i-amd punishment death imprisonment qisas diyat Section 302 PPC.

Results with rewritten query:
[1] Score=0.7419 | pakistan penal code 1860
     ordinary course of nature is likely to cause death, or with the knowledge that his act is so imminently dangerous that it must in all probability cause death, causes the death of such person, is said 

[2] Score=0.6779 | pakistan penal code 1860
     extend to ten years.
1Subs. by Act I of 2005, s. 9 for “Fourteen Years”.

321. Qatl-bis-sabab.
__ Whoever, without any intention to cause death of, or cause harm to, any person, does any unlawful act 

[3] Score=0.6768 | pakistan penal code 1860
     the 1Subs. by Act 26 of 2011, s.2

Subs. by Act XLIII of 2016, s. 6.

court, an officer authorised by the court shall give permission for the execution of qisas and the Government shall cause executio

